# Senseiver for Sparse ERA5 Reconstruction

A self-contained multivariate reconstruction experiment.

The experiment uses simulated sparse sensors and reports RMSE, normalized RMSE, latitude-weighted anomaly correlation, and windowed SSIM.

## 0. Setup

### 0.1 Import libraries and configure experiment

In [1]:
import numpy as np
import torch
import torch.nn as nn
import xarray as xr
from torch.utils.data import DataLoader, Dataset

try:
    from skimage.metrics import structural_similarity
except Exception:
    structural_similarity = None

# Config locale
ERA5_PATH = "gs://weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr"
TIME_SLICE = slice(0, 4000)
LEVEL_TARGET = 850
VAR_CANDIDATES = {
    "u": ["10m_u_component_of_wind", "u10"],
    "v": ["10m_v_component_of_wind", "v10"],
    "z": ["geopotential", "z"],
    "q": ["specific_humidity", "q"],
    "t2m": ["2m_temperature", "t2m"],
}
MULTIVAR_KEYS = ["z", "q", "t2m", "u", "v"]

KEEP_RATIO = 0.1
SEED_TRAIN = 10
SEED_VAL = 20
SEED_TEST = 30

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

BATCH_SIZE_TRAIN = 16
BATCH_SIZE_EVAL = 32
LR = 5e-3
MAX_EPOCHS = 30
PATIENCE = 5

SENSEIVER_N_LATENT = 128
SENSEIVER_D_LATENT = 256
SENSEIVER_N_HEADS = 8
SENSEIVER_N_SELF_ATTN_LAYERS = 4
SENSEIVER_DROPOUT = 0.1

# Active la reconstruction Voronoi complète en amont de Senseiver.
# Chaque sample retourne alors N = H*W tokens (toute la grille).
SENSEIVER_USE_FULL_VORONOI = True

WEIGHT_UNKNOWN = 0.8
WEIGHT_ALL = 0.2
LAMBDA_GRAD = 1.0

# Protocole d'entraînement aligné reconstruction: poids fixes par défaut.
MULTIVAR_WEIGHTS = [1, 1, 1, 1, 1]  # (z, q, t2m, u, v)
FIXED_LOSS_WEIGHTS = None            # None -> uniformes ; sinon ex: MULTIVAR_WEIGHTS



### 0.2 Install data dependencies if needed

In [2]:
%pip -q install zarr fsspec gcsfs scikit-image
print("Dépendances installées. Si l'erreur persiste, redémarre le kernel puis relance depuis la cellule Imports.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 124.7 MB/s eta 0:00:0000:01
Dépendances installées. Si l'erreur persiste, redémarre le kernel puis relance depuis la cellule Imports.


## 1. Data and sensors

### 1.1 Define ERA5 loading and preprocessing

In [3]:
def open_dataset() -> xr.Dataset:
    try:
        return xr.open_zarr(ERA5_PATH, storage_options={"token": "anon"})
    except ValueError as e:
        msg = str(e)
        if "unrecognized engine 'zarr'" in msg or "engine 'zarr'" in msg:
            raise RuntimeError(
                "L'engine zarr n'est pas installé. Exécute d'abord la cellule 'Installation des dépendances I/O (Colab)', "
                "puis redémarre le kernel et relance le notebook depuis le début."
            ) from e
        raise


def pick_first_existing(candidates: list[str], ds: xr.Dataset) -> str | None:
    for c in candidates:
        if c in ds.data_vars:
            return c
    return None


def find_variables(ds: xr.Dataset, keys: list[str] | None = None) -> dict[str, str | None]:
    if keys is None:
        keys = list(VAR_CANDIDATES.keys())
    return {k: pick_first_existing(VAR_CANDIDATES[k], ds) for k in keys}


def to_THW(da: xr.DataArray, level_target: int = LEVEL_TARGET, time_slice: slice = TIME_SLICE) -> np.ndarray:
    if "level" in da.dims:
        da = da.sel(level=level_target, method="nearest")
    if "time" not in da.dims:
        raise ValueError(f"{da.name}: pas de dimension time. dims={da.dims}")
    da = da.isel(time=time_slice)

    lat_dim = next((d for d in da.dims if d in ("lat", "latitude")), None)
    lon_dim = next((d for d in da.dims if d in ("lon", "longitude")), None)
    if lat_dim is None or lon_dim is None:
        raise ValueError(f"{da.name}: dimension lat/lon introuvable. dims={da.dims}")

    da = da.transpose("time", lat_dim, lon_dim)
    arr = da.astype("float32").values
    if arr.ndim != 3:
        raise ValueError(f"{da.name}: attendu 3D, obtenu shape={arr.shape}")
    return arr


def get_lat_weights_2d(ds: xr.Dataset) -> np.ndarray:
    """Poids latitude cos(lat), normalisés à moyenne 1, shape (H, W)."""
    lat_name = "lat" if "lat" in ds.coords else ("latitude" if "latitude" in ds.coords else None)
    lon_name = "lon" if "lon" in ds.coords else ("longitude" if "longitude" in ds.coords else None)
    if lat_name is None or lon_name is None:
        raise RuntimeError("Coordonnées lat/lon introuvables pour calculer les poids latitude.")

    lat = np.asarray(ds[lat_name].values, dtype=np.float32)
    lon = np.asarray(ds[lon_name].values, dtype=np.float32)
    w_lat = np.cos(np.deg2rad(lat)).astype(np.float32)
    w_lat = np.clip(w_lat, 1e-6, None)

    w2d = np.repeat(w_lat[:, None], len(lon), axis=1)
    w2d = w2d / (float(np.mean(w2d)) + 1e-12)
    return w2d.astype(np.float32)


def load_fields(ds: xr.Dataset, keys: list[str]) -> dict[str, np.ndarray]:
    vars_found = find_variables(ds, keys)
    fields = {}
    for k, vname in vars_found.items():
        if vname is None:
            print(f"[loader] Variable '{k}' introuvable, ignorée.")
            continue
        fields[k] = to_THW(ds[vname])
        print(f"[loader] {k} ({vname}): {fields[k].shape}")
    return fields


def compute_split_indices(n: int, train_ratio: float = TRAIN_RATIO, val_ratio: float = VAL_RATIO):
    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)
    idx_train = np.arange(0, n_train)
    idx_val = np.arange(n_train, n_train + n_val)
    idx_test = np.arange(n_train + n_val, n)
    return idx_train, idx_val, idx_test


def compute_stats(fields: dict[str, np.ndarray], idx_train: np.ndarray):
    means, stds = {}, {}
    for k, arr in fields.items():
        means[k] = float(arr[idx_train].mean())
        stds[k] = float(arr[idx_train].std() + 1e-6)
        print(f"[loader] {k}: mu={means[k]:.4f}, sigma={stds[k]:.4f}")
    return means, stds


def build_climatology_maps(fields: dict[str, np.ndarray], keys: list[str], idx_train: np.ndarray) -> dict[str, np.ndarray]:
    """Climatologie 2D par variable sur split train (pour ACC)."""
    maps = {}
    for k in keys:
        maps[k] = fields[k][idx_train].mean(axis=0).astype(np.float32)
    return maps


### 1.2 Define sensor masks and dataset

In [4]:
def make_sparse_mask(h: int, w: int, keep_ratio: float = KEEP_RATIO, rng=None) -> np.ndarray:
    rng = np.random.default_rng() if rng is None else rng
    n = h * w
    k = max(1, int(round(keep_ratio * n)))
    idx = rng.choice(n, size=k, replace=False)
    m = np.zeros(n, dtype=np.float32)
    m[idx] = 1.0
    return m.reshape(h, w)


def _voronoi_assignments(mask: np.ndarray, grid_ij: np.ndarray):
    # Plus proche capteur pour chaque pixel de la grille (style Voronoi)
    ys, xs = np.where(mask == 1)
    if len(ys) == 0:
        raise ValueError("Masque vide : aucun capteur.")

    sensors_ij = np.stack([ys, xs], axis=1).astype(np.float32)     # (N, 2)
    query_ij = grid_ij.astype(np.float32)                           # (H*W, 2)

    # Distances carrées (H*W, N), puis indice du capteur le plus proche
    d2 = ((query_ij[:, None, :] - sensors_ij[None, :, :]) ** 2).sum(axis=2)
    nn = np.argmin(d2, axis=1)                                      # (H*W,)
    return ys, xs, nn


class SenseiverDataset(Dataset):
    def __init__(
        self,
        fields: dict,
        keys: list,
        means: dict,
        stds: dict,
        indices: np.ndarray,
        keep_ratio: float = KEEP_RATIO,
        base_seed: int = 0,
        use_full_voronoi: bool = SENSEIVER_USE_FULL_VORONOI,
    ):
        self.fields = fields
        self.keys = list(keys)
        self.means = means
        self.stds = stds
        self.indices = np.array(indices, dtype=int)
        self.keep_ratio = keep_ratio
        self.base_seed = base_seed
        self.use_full_voronoi = use_full_voronoi

        any_key = self.keys[0]
        _, self.H, self.W = self.fields[any_key].shape

        gy, gx = np.meshgrid(
            np.linspace(0, 1, self.H),
            np.linspace(0, 1, self.W),
            indexing="ij",
        )
        self._coord_y = gy.astype(np.float32)
        self._coord_x = gx.astype(np.float32)

        gij_y, gij_x = np.meshgrid(np.arange(self.H), np.arange(self.W), indexing="ij")
        self._grid_ij = np.stack([gij_y.ravel(), gij_x.ravel()], axis=1).astype(np.float32)  # (H*W,2)
        self._grid_coords = np.stack([self._coord_y.ravel(), self._coord_x.ravel()], axis=1).astype(np.float32)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int):
        t = int(self.indices[i])
        rng = np.random.default_rng(self.base_seed + t)

        mask = make_sparse_mask(self.H, self.W, self.keep_ratio, rng)
        ys, xs, nn = _voronoi_assignments(mask, self._grid_ij)

        k_vars = len(self.keys)

        if self.use_full_voronoi:
            # Entrée dense: toutes les positions de grille, valeurs reconstruites Voronoi.
            sensor_vals = np.empty((self.H * self.W, k_vars), dtype=np.float32)
            for vi, k in enumerate(self.keys):
                field = self.fields[k][t]
                mu, sig = self.means[k], self.stds[k]
                obs_vals = field[ys, xs].astype(np.float32)                 # (N,)
                vor_flat = obs_vals[nn]                                     # (H*W,)
                sensor_vals[:, vi] = (vor_flat - mu) / sig

            sensors = np.concatenate([sensor_vals, self._grid_coords], axis=1)  # (H*W, K+2)
        else:
            # Entrée sparse originale: seulement les capteurs réellement observés.
            coord_y = self._coord_y[ys, xs]
            coord_x = self._coord_x[ys, xs]
            n_sensors = len(ys)

            sensor_vals = np.empty((n_sensors, k_vars), dtype=np.float32)
            for vi, k in enumerate(self.keys):
                field = self.fields[k][t]
                mu, sig = self.means[k], self.stds[k]
                sensor_vals[:, vi] = (field[ys, xs] - mu) / sig

            sensors = np.concatenate([sensor_vals, coord_y[:, None], coord_x[:, None]], axis=1)

        y_list, m_list = [], []
        for k in self.keys:
            field = self.fields[k][t]
            mu, sig = self.means[k], self.stds[k]
            y_list.append(((field - mu) / sig).astype(np.float32))
            m_list.append(mask.astype(np.float32))

        y = np.stack(y_list, axis=0)
        m = np.stack(m_list, axis=0)

        return (
            torch.tensor(sensors, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(m, dtype=torch.float32),
        )



## 2. Model and evaluation utilities

### 2.1 Define attention blocks and Senseiver

In [5]:
class CrossAttentionBlock(nn.Module):
    def __init__(self, d_query: int, d_context: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(d_query)
        self.norm_kv = nn.LayerNorm(d_context)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_query,
            num_heads=n_heads,
            kdim=d_context,
            vdim=d_context,
            dropout=dropout,
            batch_first=True,
        )
        self.norm_ff = nn.LayerNorm(d_query)
        self.ff = nn.Sequential(
            nn.Linear(d_query, d_query * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_query * 2, d_query),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, q: torch.Tensor, kv: torch.Tensor) -> torch.Tensor:
        q_norm = self.norm_q(q)
        kv_norm = self.norm_kv(kv)
        attn_out, _ = self.attn(q_norm, kv_norm, kv_norm)
        q = q + self.drop(attn_out)
        q = q + self.drop(self.ff(self.norm_ff(q)))
        return q


class SelfAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ff(self.norm2(x)))
        return x


class Senseiver(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_vars: int,
        H: int,
        W: int,
        n_latent: int = 128,
        d_latent: int = 256,
        n_heads: int = 8,
        n_self_attn_layers: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.H = H
        self.W = W
        self.out_vars = out_vars

        self.input_proj = nn.Sequential(
            nn.Linear(in_features + 2, d_latent),
            nn.LayerNorm(d_latent),
        )

        self.latent = nn.Parameter(torch.randn(1, n_latent, d_latent) * 0.02)

        self.encoder = CrossAttentionBlock(
            d_query=d_latent,
            d_context=d_latent,
            n_heads=n_heads,
            dropout=dropout,
        )

        self.processor = nn.ModuleList([
            SelfAttentionBlock(d_latent, n_heads, dropout)
            for _ in range(n_self_attn_layers)
        ])

        self.grid_proj = nn.Sequential(
            nn.Linear(2, d_latent),
            nn.LayerNorm(d_latent),
        )

        self.decoder = CrossAttentionBlock(
            d_query=d_latent,
            d_context=d_latent,
            n_heads=n_heads,
            dropout=dropout,
        )

        self.output_proj = nn.Linear(d_latent, out_vars)

        gy, gx = torch.meshgrid(
            torch.linspace(0, 1, H),
            torch.linspace(0, 1, W),
            indexing="ij",
        )
        grid_pos = torch.stack([gy.flatten(), gx.flatten()], dim=1)
        self.register_buffer("grid_pos", grid_pos)

    def forward(self, sensors: torch.Tensor) -> torch.Tensor:
        bsz = sensors.shape[0]
        s = self.input_proj(sensors)

        latent = self.latent.expand(bsz, -1, -1)
        latent = self.encoder(latent, s)

        for block in self.processor:
            latent = block(latent)

        grid = self.grid_pos.unsqueeze(0).expand(bsz, -1, -1)
        g = self.grid_proj(grid)
        g = self.decoder(g, latent)

        out = self.output_proj(g)
        out = out.permute(0, 2, 1)
        out = out.reshape(bsz, self.out_vars, self.H, self.W)
        return out


### 2.2 Define loss and training loop

In [6]:
def gradient_loss(pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    gx_pred = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    gx_gt = y[:, :, :, 1:] - y[:, :, :, :-1]
    gy_pred = pred[:, :, 1:, :] - pred[:, :, :-1, :]
    gy_gt = y[:, :, 1:, :] - y[:, :, :-1, :]
    return ((gx_pred - gx_gt) ** 2).mean() + ((gy_pred - gy_gt) ** 2).mean()


def loss_fn_mv(pred: torch.Tensor, y: torch.Tensor, m: torch.Tensor, weights: list, device) -> torch.Tensor:
    w = torch.tensor(weights, dtype=torch.float32, device=device)
    err2 = (pred - y) ** 2 * w[None, :, None, None]
    unknown = 1.0 - m
    l_unknown = (err2 * unknown).sum() / (unknown.sum() + 1e-6)
    l_all = err2.mean()
    l_mse = WEIGHT_UNKNOWN * l_unknown + WEIGHT_ALL * l_all
    return l_mse + LAMBDA_GRAD * gradient_loss(pred, y)


def print_epoch_progress(epoch: int, max_epochs: int, tr_mean: float, va_mean: float,
                         best_val: float, wait: int, patience: int,
                         keys: list, weights: list[float]):
    w_str = " | ".join(f"{k}={weights[i]:.2f}" for i, k in enumerate(keys))
    print(
        f"Epoch {epoch:02d}/{max_epochs:02d} | train={tr_mean:.5f} | val={va_mean:.5f} | "
        f"best={best_val:.5f} | patience={wait}/{patience} | poids fixes [{w_str}]",
        flush=True,
    )


def train_model_fixed(model, train_loader, val_loader, optimizer, keys: list, device,
                      max_epochs: int = 30, patience: int = 5, fixed_weights: list[float] | None = None):
    k_vars = len(keys)
    if fixed_weights is None:
        weights = [1.0] * k_vars
    else:
        if len(fixed_weights) != k_vars:
            raise ValueError(f"fixed_weights doit avoir {k_vars} valeurs (reçu {len(fixed_weights)}).")
        weights = [float(w) for w in fixed_weights]

    print(
        "[train_model_fixed] Note: val_loss comparable intra-run uniquement; inter-run, comparer plutôt NRMSE/ACC/SSIM.",
        flush=True,
    )

    best_val, best_state, wait = float("inf"), None, 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        tr_losses = []
        for x, y, m in train_loader:
            x, y, m = x.to(device), y.to(device), m.to(device)
            optimizer.zero_grad()
            loss = loss_fn_mv(model(x), y, m, weights, device)
            loss.backward()
            optimizer.step()
            tr_losses.append(loss.item())

        model.eval()
        va_losses = []
        with torch.no_grad():
            for x, y, m in val_loader:
                x, y, m = x.to(device), y.to(device), m.to(device)
                pred = model(x)
                va_losses.append(loss_fn_mv(pred, y, m, weights, device).item())

        tr_mean = float(np.mean(tr_losses))
        va_mean = float(np.mean(va_losses))

        improved = va_mean < best_val - 1e-5
        if improved:
            best_val = va_mean
            best_state = {state_key: state_val.cpu().clone() for state_key, state_val in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        print_epoch_progress(
            epoch=epoch,
            max_epochs=max_epochs,
            tr_mean=tr_mean,
            va_mean=va_mean,
            best_val=best_val,
            wait=wait,
            patience=patience,
            keys=keys,
            weights=weights,
        )

        if wait >= patience:
            print(f"Early stopping à l'epoch {epoch}.", flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
        print(f"Meilleur val loss : {best_val:.5f}", flush=True)
    return model


### 2.3 Define evaluation metrics

In [7]:
def _resolve_lat_weights(lat_weights_2d):
    if lat_weights_2d is not None:
        return np.asarray(lat_weights_2d, dtype=np.float32)
    for name in ["LAT_WEIGHTS_2D_S", "LAT_WEIGHTS_2D"]:
        if name in globals():
            return np.asarray(globals()[name], dtype=np.float32)
    raise RuntimeError("Poids latitude introuvables. Calculer get_lat_weights_2d(ds) avant l'évaluation.")


def _resolve_clim_mv(climatology_maps, keys):
    if climatology_maps is not None:
        return {k: np.asarray(climatology_maps[k], dtype=np.float32) for k in keys}
    for name in ["CLIM_MV_MAPS_S", "CLIM_MV_MAPS"]:
        if name in globals():
            src = globals()[name]
            if isinstance(src, dict) and all(k in src for k in keys):
                return {k: np.asarray(src[k], dtype=np.float32) for k in keys}
    raise RuntimeError("Climatologie multivariée introuvable. Construire build_climatology_maps(...) sur le train.")


def ssim_2d(pred: np.ndarray, gt: np.ndarray) -> float:
    """SSIM locale (fenêtre glissante), version skimage standard."""
    if structural_similarity is None:
        raise ImportError("skimage.metrics.structural_similarity indisponible. Installer scikit-image.")

    pred = np.asarray(pred, dtype=np.float32)
    gt = np.asarray(gt, dtype=np.float32)
    data_min = float(min(pred.min(), gt.min()))
    data_max = float(max(pred.max(), gt.max()))
    data_range = max(data_max - data_min, 1e-6)

    h, w = pred.shape
    max_win = min(h, w)
    if max_win % 2 == 0:
        max_win -= 1
    win = min(11, max_win)
    if win < 3:
        return float("nan")

    return float(
        structural_similarity(
            gt,
            pred,
            data_range=data_range,
            gaussian_weights=True,
            sigma=1.5,
            use_sample_covariance=False,
            win_size=win,
        )
    )


def acc_2d(pred: np.ndarray, gt: np.ndarray, climatology_map: np.ndarray, lat_weights_2d: np.ndarray) -> float:
    """ACC WeatherBench simplifié: anomalies vs climatologie + pondération cos(lat)."""
    pred = np.asarray(pred, dtype=np.float64)
    gt = np.asarray(gt, dtype=np.float64)
    clim = np.asarray(climatology_map, dtype=np.float64)
    w = np.asarray(lat_weights_2d, dtype=np.float64)

    pa = pred - clim
    ga = gt - clim

    num = np.sum(w * pa * ga)
    den = np.sqrt(np.sum(w * pa * pa) * np.sum(w * ga * ga))
    if den < 1e-12:
        return float("nan")
    return float(num / den)


def evaluate_multivariate(model, test_loader, keys: list[str], means: dict, stds: dict, device,
                          climatology_maps=None, lat_weights_2d=None) -> dict:
    """Évaluation multivariée avec ACC climatologique et SSIM locale."""
    model.eval()
    clim_maps = _resolve_clim_mv(climatology_maps, keys)
    w_lat = _resolve_lat_weights(lat_weights_2d)

    sse_all = {k: 0.0 for k in keys}
    n_all = {k: 0.0 for k in keys}
    sse_unknown = {k: 0.0 for k in keys}
    n_unknown = {k: 0.0 for k in keys}

    acc_sum = {k: 0.0 for k in keys}
    acc_n = {k: 0 for k in keys}
    ssim_sum = {k: 0.0 for k in keys}
    ssim_n = {k: 0 for k in keys}

    with torch.no_grad():
        for x, y, m in test_loader:
            x, y, m = x.to(device), y.to(device), m.to(device)
            pred = model(x)

            for i, k in enumerate(keys):
                mu, sig = means[k], stds[k]
                pred_k = pred[:, i] * sig + mu
                y_k = y[:, i] * sig + mu

                e2 = (pred_k - y_k) ** 2
                sse_all[k] += float(e2.sum().item())
                n_all[k] += float(e2.numel())

                unknown = 1.0 - m[:, i]
                sse_unknown[k] += float((e2 * unknown).sum().item())
                n_unknown[k] += float(unknown.sum().item())

                pred_np = pred_k.cpu().numpy()
                y_np = y_k.cpu().numpy()
                clim_k = clim_maps[k]
                for b in range(pred_np.shape[0]):
                    a = acc_2d(pred_np[b], y_np[b], clim_k, w_lat)
                    if np.isfinite(a):
                        acc_sum[k] += float(a)
                        acc_n[k] += 1

                    s = ssim_2d(pred_np[b], y_np[b])
                    if np.isfinite(s):
                        ssim_sum[k] += float(s)
                        ssim_n[k] += 1

    results = {}
    for k in keys:
        rmse_g = float(np.sqrt(sse_all[k] / max(n_all[k], 1.0)))
        rmse_u = float(np.sqrt(sse_unknown[k] / max(n_unknown[k], 1.0)))
        results[k] = {
            "rmse_global": rmse_g,
            "rmse_unknown": rmse_u,
            "nrmse_global": rmse_g / stds[k],
            "nrmse_unknown": rmse_u / stds[k],
            "acc": float(acc_sum[k] / max(acc_n[k], 1)),
            "ssim": float(ssim_sum[k] / max(ssim_n[k], 1)),
        }
    return results


def print_results_multivariate(results: dict):
    print("── Résultats multivarié ──────────────────────────────────────────────────────────────────")
    print(
        f"  {'Var':>4s} | {'RMSE global':>12s} | {'NRMSE global':>12s} | "
        f"{'RMSE non-obs.':>13s} | {'NRMSE non-obs.':>14s} | {'ACC':>6s} | {'SSIM':>6s}"
    )
    print("  " + "-" * 87)
    for k, v in results.items():
        print(
            f"  {k:>4s} | {v['rmse_global']:>12.4f} | {v['nrmse_global']:>12.4f} | "
            f"{v['rmse_unknown']:>13.4f} | {v['nrmse_unknown']:>14.4f} | "
            f"{v['acc']:>6.4f} | {v['ssim']:>6.4f}"
        )
    nrmse_global_mean = float(np.mean([v["nrmse_global"] for v in results.values()]))
    nrmse_unknown_mean = float(np.mean([v["nrmse_unknown"] for v in results.values()]))
    acc_mean = float(np.nanmean([v["acc"] for v in results.values()]))
    ssim_mean = float(np.mean([v["ssim"] for v in results.values()]))
    print("  " + "-" * 87)
    print(
        f"  {'moy.':>4s} | {'':12} | {nrmse_global_mean:>12.4f} | {'':13} | "
        f"{nrmse_unknown_mean:>14.4f} | {acc_mean:>6.4f} | {ssim_mean:>6.4f}"
    )


## 3. Run the experiment

### 3.1 Load ERA5 fields

In [8]:
ds_wb2 = open_dataset()
LAT_WEIGHTS_2D_S = get_lat_weights_2d(ds_wb2)
fields = load_fields(ds_wb2, MULTIVAR_KEYS)

keys = list(fields.keys())
if len(keys) == 0:
    raise RuntimeError("Aucune variable disponible pour Senseiver.")

print("Variables utilisées:", keys)
print(f"LAT_WEIGHTS_2D_S: shape={LAT_WEIGHTS_2D_S.shape}")


[loader] z (geopotential): (4000, 32, 64)
[loader] q (specific_humidity): (4000, 32, 64)
[loader] t2m (2m_temperature): (4000, 32, 64)
[loader] u (10m_u_component_of_wind): (4000, 32, 64)
[loader] v (10m_v_component_of_wind): (4000, 32, 64)
Variables utilisées: ['z', 'q', 't2m', 'u', 'v']
LAT_WEIGHTS_2D_S: shape=(32, 64)


### 3.2 Split data and compute training statistics

In [9]:
N = next(iter(fields.values())).shape[0]
idx_train_s, idx_val_s, idx_test_s = compute_split_indices(N)
means_s, stds_s = compute_stats(fields, idx_train_s)
CLIM_MV_MAPS_S = build_climatology_maps(fields, keys, idx_train_s)

print(f"T total={N} | train={len(idx_train_s)} | val={len(idx_val_s)} | test={len(idx_test_s)}")
print(f"Climatologie calculée sur split train pour {len(CLIM_MV_MAPS_S)} variables.")


[loader] z: mu=13742.5244, sigma=1421.1805
[loader] q: mu=0.0045, sigma=0.0039
[loader] t2m: mu=278.0344, sigma=21.3680
[loader] u: mu=-0.0598, sigma=4.8730
[loader] v: mu=0.1973, sigma=4.1407
T total=4000 | train=2800 | val=600 | test=600
Climatologie calculée sur split train pour 5 variables.


### 3.3 Build datasets and data loaders

In [10]:
train_senseiver = SenseiverDataset(
    fields,
    keys,
    means_s,
    stds_s,
    idx_train_s,
    keep_ratio=KEEP_RATIO,
    base_seed=SEED_TRAIN,
    use_full_voronoi=SENSEIVER_USE_FULL_VORONOI,
)
val_senseiver = SenseiverDataset(
    fields,
    keys,
    means_s,
    stds_s,
    idx_val_s,
    keep_ratio=KEEP_RATIO,
    base_seed=SEED_VAL,
    use_full_voronoi=SENSEIVER_USE_FULL_VORONOI,
)
test_senseiver = SenseiverDataset(
    fields,
    keys,
    means_s,
    stds_s,
    idx_test_s,
    keep_ratio=KEEP_RATIO,
    base_seed=SEED_TEST,
    use_full_voronoi=SENSEIVER_USE_FULL_VORONOI,
)

train_loader_senseiver = DataLoader(train_senseiver, batch_size=BATCH_SIZE_TRAIN, shuffle=True, num_workers=0)
val_loader_senseiver = DataLoader(val_senseiver, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=0)
test_loader_senseiver = DataLoader(test_senseiver, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=0)

x0, y0, m0 = train_senseiver[0]
mode_name = "Voronoi full-grid" if SENSEIVER_USE_FULL_VORONOI else "Sparse sensors"
print(f"Dataset sizes | train={len(train_senseiver)} val={len(val_senseiver)} test={len(test_senseiver)}")
print(f"Input mode={mode_name} | x0={tuple(x0.shape)} y0={tuple(y0.shape)} m0={tuple(m0.shape)}")



Dataset sizes | train=2800 val=600 test=600
Input mode=Voronoi full-grid | x0=(2048, 7) y0=(5, 32, 64) m0=(5, 32, 64)


### 3.4 Initialize Senseiver and optimizer

In [11]:
any_key = keys[0]
_, H_s, W_s = fields[any_key].shape

device_senseiver = "cuda" if torch.cuda.is_available() else "cpu"

senseiver_model = Senseiver(
    in_features=len(keys),
    out_vars=len(keys),
    H=H_s,
    W=W_s,
    n_latent=SENSEIVER_N_LATENT,
    d_latent=SENSEIVER_D_LATENT,
    n_heads=SENSEIVER_N_HEADS,
    n_self_attn_layers=SENSEIVER_N_SELF_ATTN_LAYERS,
    dropout=SENSEIVER_DROPOUT,
).to(device_senseiver)

optimizer_senseiver = torch.optim.Adam(senseiver_model.parameters(), lr=LR)

n_params_senseiver = sum(p.numel() for p in senseiver_model.parameters() if p.requires_grad)
print(f"Senseiver | K={len(keys)} | H={H_s} W={W_s} | {n_params_senseiver:,} paramètres | device={device_senseiver}")


Senseiver | K=5 | H=32 W=64 | 3,201,541 paramètres | device=cuda


### 3.5 Train Senseiver

In [12]:
if FIXED_LOSS_WEIGHTS is None:
    print("[Senseiver] MULTIVAR_WEIGHTS défini mais non utilisé (FIXED_LOSS_WEIGHTS=None).")
else:
    if len(FIXED_LOSS_WEIGHTS) != len(keys):
        raise ValueError(
            f"Incompatibilité FIXED_LOSS_WEIGHTS: len={len(FIXED_LOSS_WEIGHTS)} vs K={len(keys)}"
        )

senseiver_model = train_model_fixed(
    senseiver_model,
    train_loader_senseiver,
    val_loader_senseiver,
    optimizer_senseiver,
    keys=keys,
    device=device_senseiver,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    fixed_weights=FIXED_LOSS_WEIGHTS,
)



[Senseiver] MULTIVAR_WEIGHTS défini mais non utilisé (FIXED_LOSS_WEIGHTS=None).
[train_model_fixed] Note: val_loss comparable intra-run uniquement; inter-run, comparer plutôt NRMSE/ACC/SSIM.
Epoch 01/30 | train=1.79116 | val=1.04409 | best=1.04409 | patience=0/5 | poids fixes [z=1.00 | q=1.00 | t2m=1.00 | u=1.00 | v=1.00]
Epoch 02/30 | train=1.03099 | val=0.96360 | best=0.96360 | patience=0/5 | poids fixes [z=1.00 | q=1.00 | t2m=1.00 | u=1.00 | v=1.00]
Epoch 03/30 | train=0.95400 | val=0.91626 | best=0.91626 | patience=0/5 | poids fixes [z=1.00 | q=1.00 | t2m=1.00 | u=1.00 | v=1.00]
Epoch 04/30 | train=0.92904 | val=0.91563 | best=0.91563 | patience=0/5 | poids fixes [z=1.00 | q=1.00 | t2m=1.00 | u=1.00 | v=1.00]
Epoch 05/30 | train=0.91531 | val=0.91396 | best=0.91396 | patience=0/5 | poids fixes [z=1.00 | q=1.00 | t2m=1.00 | u=1.00 | v=1.00]
Epoch 06/30 | train=0.90129 | val=0.88528 | best=0.88528 | patience=0/5 | poids fixes [z=1.00 | q=1.00 | t2m=1.00 | u=1.00 | v=1.00]
Epoch 07/30

### 3.6 Evaluate on the test split

In [13]:
senseiver_results = evaluate_multivariate(
    senseiver_model,
    test_loader_senseiver,
    keys,
    means_s,
    stds_s,
    device_senseiver,
    climatology_maps=CLIM_MV_MAPS_S,
    lat_weights_2d=LAT_WEIGHTS_2D_S,
)
print_results_multivariate(senseiver_results)


── Résultats multivarié ──────────────────────────────────────────────────────────────────
   Var |  RMSE global | NRMSE global | RMSE non-obs. | NRMSE non-obs. |    ACC |   SSIM
  ---------------------------------------------------------------------------------------
     z |     544.4655 |       0.3831 |      544.7491 |         0.3833 | 0.3994 | 0.7349
     q |       0.0014 |       0.3546 |        0.0014 |         0.3548 | 0.5734 | 0.6567
   t2m |       3.9966 |       0.1870 |        3.9962 |         0.1870 | 0.7292 | 0.8718
     u |       3.4765 |       0.7134 |        3.4786 |         0.7139 | 0.3190 | 0.3796
     v |       3.6538 |       0.8824 |        3.6548 |         0.8827 | 0.2365 | 0.1940
  ---------------------------------------------------------------------------------------
  moy. |              |       0.5041 |               |         0.5043 | 0.4515 | 0.5674
